In [2]:
import pandas as pd

In [15]:
og_data = pd.read_csv("food_delivery_raw.csv")
data = og_data.copy()

In [16]:
data.columns

Index(['order_id', 'order_item_id', 'order_ts', 'customer_id', 'customer_age',
       'customer_gender', 'customer_area', 'customer_signup_date',
       'restaurant_id', 'restaurant_name', 'restaurant_area',
       'restaurant_cuisine', 'restaurant_rating', 'item_id', 'item_name',
       'category', 'quantity', 'unit_price', 'line_subtotal',
       'delivery_distance_km', 'estimated_delivery_min', 'actual_delivery_min',
       'delivery_status', 'discount', 'delivery_fee', 'packaging_fee', 'gst',
       'tip', 'final_amount', 'payment_method', 'order_status', 'food_rating',
       'delivery_rating'],
      dtype='str')

In [18]:
pd.set_option("display.max_columns",None)

In [20]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21252 entries, 0 to 21251
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   order_id                21252 non-null  str    
 1   order_item_id           21252 non-null  str    
 2   order_ts                21252 non-null  str    
 3   customer_id             21252 non-null  int64  
 4   customer_age            21252 non-null  int64  
 5   customer_gender         21252 non-null  str    
 6   customer_area           21252 non-null  str    
 7   customer_signup_date    21252 non-null  str    
 8   restaurant_id           21252 non-null  int64  
 9   restaurant_name         21252 non-null  str    
 10  restaurant_area         21252 non-null  str    
 11  restaurant_cuisine      21252 non-null  str    
 12  restaurant_rating       20916 non-null  float64
 13  item_id                 21252 non-null  int64  
 14  item_name               21141 non-null  str    
 

In [25]:
data.columns

Index(['order_id', 'order_item_id', 'order_ts', 'customer_id', 'customer_age',
       'customer_gender', 'customer_area', 'customer_signup_date',
       'restaurant_id', 'restaurant_name', 'restaurant_area',
       'restaurant_cuisine', 'restaurant_rating', 'item_id', 'item_name',
       'category', 'quantity', 'unit_price', 'line_subtotal',
       'delivery_distance_km', 'estimated_delivery_min', 'actual_delivery_min',
       'delivery_status', 'discount', 'delivery_fee', 'packaging_fee', 'gst',
       'tip', 'final_amount', 'payment_method', 'order_status', 'food_rating',
       'delivery_rating'],
      dtype='str')

# Data Transformation

In [190]:
def transform_data():
    def Dim_customer():
        customer = data[["customer_id","customer_age","customer_gender","customer_area"]].copy()
        customer["customer_area"] = customer["customer_area"].str.strip().str.title()
        customer = customer.drop_duplicates(subset = "customer_id")
        customer = customer.sort_values(by = "customer_id")

        return customer

    def Dim_restaurant():
        restaurant = data[["restaurant_id","restaurant_name","restaurant_area","restaurant_cuisine","restaurant_rating"]].copy()
        
        restaurant["restaurant_name"] = restaurant["restaurant_name"].str.strip().str.title()
        restaurant["restaurant_area"] = restaurant["restaurant_area"].str.strip().str.title()
        restaurant["restaurant_cuisine"] = restaurant["restaurant_cuisine"].str.strip().str.title()
        restaurant["restaurant_rating"] = restaurant["restaurant_rating"].astype(float)
        
        restaurant = restaurant.drop_duplicates(subset = "restaurant_id")
        restaurant = restaurant.sort_values(by = "restaurant_id")

        return restaurant

    def Dim_item():
        items = data[["item_id","restaurant_id","item_name","category"]].copy()
        
        items["item_name"] = (
            items.groupby("item_id")["item_name"]
                .transform(lambda x: x.ffill().bfill())
            )
        items["item_name"] = items["item_name"].str.strip().str.title()
        items["category"] = items["category"].str.strip().str.title()
        
        items = items.drop_duplicates(subset = "item_id")
        items = items.sort_values(by = "item_id")

        return items
        
    def Dim_date():
        date_time = pd.to_datetime(data["order_ts"], errors="coerce")
    
        date_df = pd.DataFrame()
    
        date_df["date"] = date_time.dt.normalize()
        date_df["date_id"] = date_time.dt.strftime("%Y%m%d")
    
        date_df["year"] = date_time.dt.year
        date_df["month"] = date_time.dt.month
        date_df["month_name"] = date_time.dt.month_name()
        date_df["day"] = date_time.dt.day
        date_df["day_name"] = date_time.dt.day_name()
        date_df["quarter"] = date_time.dt.quarter
        
        date_df = date_df.dropna(subset=["date"])
        date_df = date_df.drop_duplicates(subset=["date_id"])
    
        date_df = date_df.sort_values(by="date").reset_index(drop=True)
    
        date_df["date_id"] = date_df["date_id"].astype("int64")
    
        return date_df

    def Fact_orders():
        order_columns = ["order_id","customer_id","restaurant_id","discount","delivery_fee",
                         "packaging_fee","gst","tip","final_amount","estimated_delivery_min","actual_delivery_min",
                         "order_status","payment_method","food_rating","delivery_rating","order_ts"]
        orders = data[order_columns].copy()
        orders_date =  pd.to_datetime(data["order_ts"], errors = "coerce")
        
        orders["date_id"] = orders_date.dt.strftime("%Y%m%d").astype(int)

        orders["discount"] = orders["discount"].astype(float)
        orders["delivery_fee"] = orders["delivery_fee"].astype("Int64")
        orders["packaging_fee"] = orders["packaging_fee"].astype("Int64")
        orders["tip"] = orders["tip"].astype("Int64")
        orders["final_amount"] = orders["final_amount"].astype(float)
        orders["estimated_delivery_min"] = orders["estimated_delivery_min"].astype("Int64")
        orders["actual_delivery_min"] = orders["actual_delivery_min"].astype("Int64")
        orders["order_status"] = orders["order_status"].str.strip().str.title()
        orders["payment_method"] = orders["payment_method"].str.strip().str.title()
        orders["order_ts"] = pd.to_datetime(orders["order_ts"], errors="coerce")
        
        orders = orders.drop_duplicates(subset = "order_id")
        orders = orders[["order_id","date_id","customer_id","restaurant_id","order_ts","discount","delivery_fee",
                         "packaging_fee","gst","tip","final_amount","estimated_delivery_min","actual_delivery_min",
                         "order_status","payment_method","food_rating","delivery_rating"]]
        orders = orders.sort_values(by = "order_id")
        
        return orders

    def Fact_order_items():
        item_columns = [
                "order_item_id",
                "order_id",
                "item_id",
                "quantity",
                "unit_price",
            ]
        
        order_items = data[item_columns].copy()

        order_items["line_subtotal"] = order_items["quantity"] * order_items["unit_price"]
        
        order_items = order_items.drop_duplicates(subset = "order_item_id")
        order_items = order_items.sort_values(by = "order_item_id")
            
        return order_items
            
    order_items = Fact_order_items()
    orders = Fact_orders()
    customers = Dim_customer()
    restaurants = Dim_restaurant()
    items = Dim_item()
    dates = Dim_date()

    return customers, restaurants, items, dates, orders, order_items

customer_df, restaurant_df, item_df, date_df, orders_df, order_items_df = transform_data()

In [192]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21252 entries, 0 to 21251
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_item_id  21252 non-null  str    
 1   order_id       21252 non-null  str    
 2   item_id        21252 non-null  int64  
 3   quantity       21252 non-null  int64  
 4   unit_price     21252 non-null  float64
 5   line_subtotal  21252 non-null  float64
dtypes: float64(2), int64(2), str(2)
memory usage: 1.4 MB


In [193]:
orders_df[[
    "discount",
    "delivery_fee",
    "packaging_fee",
    "tip"
]].isna().sum()

discount         0
delivery_fee     0
packaging_fee    0
tip              0
dtype: int64

In [176]:
order_items_df

,order_item_id,order_id,item_id,quantity,unit_price,line_subtotal
0,OI00000101,ORD000001,360,1,100.49,100.49
1,OI00000102,ORD000001,358,3,275.44,826.32
2,OI00000103,ORD000001,357,1,236.98,236.98
3,OI00000201,ORD000002,51,2,163.64,327.28
4,OI00000301,ORD000003,329,1,222.98,222.98
...,...,...,...,...,...,...
21247,OI01199801,ORD011998,149,1,249.72,249.72
21248,OI01199802,ORD011998,150,1,278.16,278.16
21249,OI01199901,ORD011999,390,1,194.86,194.86
21250,OI01199902,ORD011999,389,1,191.00,191.00


## Data Quality Check (or) Data Validation

In [107]:
print(customer_df["customer_id"].is_unique)
print(restaurant_df["restaurant_id"].is_unique)
print(item_df["item_id"].is_unique)
print(date_df["date_id"].is_unique)

True
True
True
True


In [116]:
item_df.info()

<class 'pandas.DataFrame'>
Index: 400 entries, 564 to 183
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   item_id        400 non-null    int64
 1   restaurant_id  400 non-null    int64
 2   item_name      400 non-null    str  
 3   category       400 non-null    str  
dtypes: int64(2), str(2)
memory usage: 24.7 KB


In [118]:
print(item_df["item_id"].is_unique)
print(item_df.isnull().sum())
item_df.groupby("item_id")["restaurant_id"].nunique().max()

True
item_id          0
restaurant_id    0
item_name        0
category         0
dtype: int64


np.int64(1)

In [109]:
customer_df.loc[
    customer_df["customer_id"] == 16,
    ["customer_id", "customer_area"]
].drop_duplicates()

,customer_id,customer_area
1242,16,Miyapur


In [69]:
def date_table():
    date_time = pd.to_datetime(data["order_ts"], errors="coerce")

    date_df = pd.DataFrame()

    date_df["date"] = date_time.dt.normalize()
    date_df["date_id"] = date_time.dt.strftime("%Y%m%d")

    date_df["year"] = date_time.dt.year
    date_df["month"] = date_time.dt.month
    date_df["month_name"] = date_time.dt.month_name()
    date_df["day"] = date_time.dt.day
    date_df["day_name"] = date_time.dt.day_name()
    date_df["quarter"] = date_time.dt.quarter

    # Remove invalid dates and duplicate calendar dates
    date_df = date_df.dropna(subset=["date"])
    date_df = date_df.drop_duplicates(subset=["date_id"])

    date_df = date_df.sort_values(by="date").reset_index(drop=True)

    date_df["date_id"] = date_df["date_id"].astype("int64")

    return date_df
date_table()

,date,date_id,year,month,month_name,day,day_name,quarter
0,2025-01-01,20250101,2025,1,January,1,Wednesday,1
1,2025-01-02,20250102,2025,1,January,2,Thursday,1
2,2025-01-03,20250103,2025,1,January,3,Friday,1
3,2025-01-04,20250104,2025,1,January,4,Saturday,1
4,2025-01-05,20250105,2025,1,January,5,Sunday,1
...,...,...,...,...,...,...,...,...
360,2025-12-27,20251227,2025,12,December,27,Saturday,4
361,2025-12-28,20251228,2025,12,December,28,Sunday,4
362,2025-12-29,20251229,2025,12,December,29,Monday,4
363,2025-12-30,20251230,2025,12,December,30,Tuesday,4


# Loading Data

In [203]:
import os
from sqlalchemy import create_engine, URL
from sqlalchemy import text

connection_url = URL.create(
    "mysql+mysqlconnector",
    username="root",
    password="kinnu",
    host="localhost",
    port=3306,
    database="food_delivery_db"
)
engine = create_engine(connection_url)

customer_df.to_sql(name="dim_customers",
                   con=engine,
                   if_exists='append',
                   index=False)

restaurant_df.to_sql(name="dim_restaurants",
                   con=engine,
                   if_exists='append',
                   index=False)



80

In [204]:
date_df.to_sql(name="dim_date",
                   con=engine,
                   if_exists='append',
                   index=False)

item_df.to_sql(name="dim_items",
                   con=engine,
                   if_exists='append',
                   index=False)

orders_df.to_sql(name="fact_orders",
                   con=engine,
                   if_exists='append',
                   index=False)

order_items_df.to_sql(name="fact_order_items",
                   con=engine,
                   if_exists='append',
                   index=False)

21252